In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Model Identification
MODEL_NAME = "btrabucco/Insta-Qwen3-1.7B-SFT"

# 2. Load Model and Tokenizer
# We assume the environment supports bfloat16 for the Qwen model.
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,  # Use bfloat16 as mentioned in InSTA training details [4]
    ).to("cuda")
    model.eval()
except ImportError:
    print("Please ensure you have PyTorch and the transformers library installed.")
    exit()

# 3. Construct the Agent Prompt
# The agent input is multimodal: Instruction (c) + Context (Enc(st)) [5].
# The prompt must follow the structure defined in the agent system prompt [6-17].

# We simulate the prompt for the task: "Browse fonts suitable for a children’s book." [2, 18]
# Note: In a real scenario, the "webpage and actions" section would include the Markdown-formatted HTML [5].

AGENT_SYSTEM_PROMPT = """
You are helping me complete tasks by operating a web browser. I will share the current task, and a sequence of webpages and actions from previous steps.

## Action Instructions
Based on the information we discovered so far, and the progress we made in previous steps, you are helping me determine the next action.
You will provide an action as JSON in a fenced code block:
```json {
"action_key": str, "action_kwargs": dict, "target_element_id": int | null
} ```
... [Action Definitions Omitted for Brevity] ...
## Formatting Your Response
Write a 200 word revised plan based on new information we discovered, and progress we made in previous steps. After your response, provide the next action as JSON in a fenced code block.
"""

TASK_INSTRUCTION = "Book a winery tour in Napa Valley in a winery which serves Mediterranean cuisine with wine testing for 4 guests on April 15, 10 am in a outdoor setup"
PAGE_HTML = """<html backend_node_id="117">\n  <body>\n    <div backend_node_id="561">\n      <div backend_node_id="562">\n        <div backend_node_id="569">\n          <div backend_node_id="570">\n            <a backend_node_id="572">\n                <text backend_node_id="573">Skip to main content</text>\n              </a>\n            <a backend_node_id="578">\n                      <text backend_node_id="579">Use Tock at your business</text>\n                    </a>\n                  <header backend_node_id="584">\n              <div backend_node_id="586">\n                  <a backend_node_id="589" aria_label="Tock home page"/>\n                  <div backend_node_id="597">\n                    <button backend_node_id="598" aria_label="Search"/>\n                    <button backend_node_id="602" aria_label="Menu"/>\n                  </div>\n                </div>\n              <div backend_node_id="607">\n                <div backend_node_id="608">\n                  <div backend_node_id="610">\n                    <a backend_node_id="611" aria_label="Tock home page"/>\n                    <button backend_node_id="619" aria_label="Book a reservation. Toggle open a menu of reservation types">\n                      <span backend_node_id="621">\n                        <text backend_node_id="622">Book a reservation</text>\n                      </span>\n                    </button>\n                    <button backend_node_id="626" aria_label="Book a reservation. Toggle open a menu of reservation types">\n                      <span backend_node_id="627">\n                        <text backend_node_id="628">Reservations</text>\n                      </span>\n                    </button>\n                  </div>\n                  <div backend_node_id="632">\n                    <button backend_node_id="633">\n                      <span backend_node_id="638">\n                        <text backend_node_id="639">Search</text>\n                      </span>\n                    </button>\n                    <button backend_node_id="640" aria_label="Search"/>\n                    <button backend_node_id="644" aria_label="Toggle menu open">\n                      <img backend_node_id="102" alt="Profile image"/>\n                    </button>\n                  </div>\n                </div>\n              </div>\n            </header>\n            <main backend_node_id="647">\n              <div backend_node_id="648">\n                <div backend_node_id="649">\n                  <div backend_node_id="108">\n                    <div backend_node_id="650">\n                      <div backend_node_id="651">\n                        <p backend_node_id="652">\n                          <text backend_node_id="653">DELICIOUS</text>\n                        </p>\n                        <p backend_node_id="654">\n                          <text backend_node_id="655">STARTS</text>\n                        </p>\n                        <p backend_node_id="656">\n                          <text backend_node_id="657">HERE.</text>\n                        </p>\n                      </div>\n                    </div>\n                  </div>\n                  <div backend_node_id="663">\n                    <div backend_node_id="664">\n                      <div backend_node_id="665">\n                        <label backend_node_id="666">\n                          <text backend_node_id="667">Reservation type</text>\n                        </label>\n                        <div backend_node_id="668">\n                          <select backend_node_id="708" name="type">\n                            <option backend_node_id="709" value="reservations" option_selected="true">\n                              <text backend_node_id="710">Dine in</text>\n                            </option>\n                            <option backend_node_id="711" value="pickup">\n                              <text backend_node_id="712">Pickup</text>\n                            </option>\n                            <option backend_node_id="713" value="delivery">\n                              <text backend_node_id="714">Delivery</text>\n                            </option>\n                            <option backend_node_id="715" value="events">\n                              <text backend_node_id="716">Events</text>\n                            </option>\n                            <option backend_node_id="717" value="wineries">\n                              <text backend_node_id="718">Wineries</text>\n                            </option>\n                            <option backend_node_id="719" value="all">\n                              <text backend_node_id="720">Everything</text>\n                            </option>\n                          </select>\n                        </div>\n                      </div>\n                      <div backend_node_id="722">\n                        <label backend_node_id="723">\n                          <text backend_node_id="724">Location</text>\n                        </label>\n                        <div backend_node_id="725">\n                          <input backend_node_id="726" name="location" type="text" value="Columbus, OH" input_value="Columbus, OH"/>\n                          <div backend_node_id="727">\n                            <button backend_node_id="728" type="button" aria_label="Select location">\n                              <svg backend_node_id="110" class="MuiSvgIcon-root"/>\n                              </button>\n                          </div>\n                        </div>\n                      </div>\n                      <div backend_node_id="732">\n                        <div backend_node_id="733">\n                          <label backend_node_id="735">\n                            <text backend_node_id="736">Date</text>\n                          </label>\n                          <div backend_node_id="737">\n                            <input backend_node_id="739" type="text" name="date" value="Sat, Mar 18" input_value="Sat, Mar 18"/>\n                            <button backend_node_id="741" type="button" aria_label="Date, selected value is Sat, Mar 18"/>\n                            </div>\n                        </div>\n                      </div>\n                      <div backend_node_id="745">\n                        <label backend_node_id="747">\n                          <text backend_node_id="748">Time</text>\n                        </label>\n                        <div backend_node_id="749">\n                          <select backend_node_id="752" name="time">\n                            <option backend_node_id="753" value="11:00" option_selected="true">\n                              <text backend_node_id="754">Now</text>\n                            </option>\n                            <option backend_node_id="755" value="11:30">\n                              <text backend_node_id="756">11:30 AM</text>\n                            </option>\n                            <option backend_node_id="757" value="12:00">\n                              <text backend_node_id="758">12:00 PM</text>\n                            </option>\n                            <option backend_node_id="759" value="12:30">\n                              <text backend_node_id="760">12:30 PM</text>\n                            </option>\n                            <option backend_node_id="761" value="13:00">\n                              <text backend_node_id="762">1:00 PM</text>\n                            </option>\n                            <option backend_node_id="763" value="13:30">\n                              <text backend_node_id="764">1:30 PM</text>\n                            </option>\n                            <option backend_node_id="765" value="14:00">\n                              <text backend_node_id="766">2:00 PM</text>\n                            </option>\n                            <option backend_node_id="767" value="14:30">\n                              <text backend_node_id="768">2:30 PM</text>\n                            </option>\n                            <option backend_node_id="769" value="15:00">\n                              <text backend_node_id="770">3:00 PM</text>\n                            </option>\n                            <option backend_node_id="771" value="15:30">\n                              <text backend_node_id="772">3:30 PM</text>\n                            </option>\n                            <option backend_node_id="773" value="16:00">\n                              <text backend_node_id="774">4:00 PM</text>\n                            </option>\n                            <option backend_node_id="775" value="16:30">\n                              <text backend_node_id="776">4:30 PM</text>\n                            </option>\n                            <option backend_node_id="777" value="17:00">\n                              <text backend_node_id="778">5:00 PM</text>\n                            </option>\n                            <option backend_node_id="779" value="17:30">\n                              <text backend_node_id="780">5:30 PM</text>\n                            </option>\n                            <option backend_node_id="781" value="18:00">\n                              <text backend_node_id="782">6:00 PM</text>\n                            </option>\n                            <option backend_node_id="783" value="18:30">\n                              <text backend_node_id="784">6:30 PM</text>\n                            </option>\n                            <option backend_node_id="785" value="19:00">\n                              <text backend_node_id="786">7:00 PM</text>\n                            </option>\n                            <option backend_node_id="787" value="19:30">\n                              <text backend_node_id="788">7:30 PM</text>\n                            </option>\n                            <option backend_node_id="789" value="20:00">\n                              <text backend_node_id="790">8:00 PM</text>\n                            </option>\n                            <option backend_node_id="791" value="20:30">\n                              <text backend_node_id="792">8:30 PM</text>\n                            </option>\n                            <option backend_node_id="793" value="21:00">\n                              <text backend_node_id="794">9:00 PM</text>\n                            </option>\n                            <option backend_node_id="795" value="21:30">\n                              <text backend_node_id="796">9:30 PM</text>\n                            </option>\n                            <option backend_node_id="797" value="22:00">\n                              <text backend_node_id="798">10:00 PM</text>\n                            </option>\n                            <option backend_node_id="799" value="22:30">\n                              <text backend_node_id="800">10:30 PM</text>\n                            </option>\n                            <option backend_node_id="801" value="23:00">\n                              <text backend_node_id="802">11:00 PM</text>\n                            </option>\n                            <option backend_node_id="803" value="23:30">\n                              <text backend_node_id="804">11:30 PM</text>\n                            </option>\n                          </select>\n                        </div>\n                      </div>\n                      <div backend_node_id="806">\n                        <label backend_node_id="808">\n                          <text backend_node_id="809">Party size</text>\n                        </label>\n                        <div backend_node_id="810">\n                          <select backend_node_id="813" name="guests">\n                            <option backend_node_id="814" value="1">\n                              <text backend_node_id="815">1 guest</text>\n                            </option>\n                            <option backend_node_id="816" value="2" option_selected="true">\n                              <text backend_node_id="817">2 guests</text>\n                            </option>\n                            <option backend_node_id="818" value="3">\n                              <text backend_node_id="819">3 guests</text>\n                            </option>\n                            <option backend_node_id="820" value="4">\n                              <text backend_node_id="821">4 guests</text>\n                            </option>\n                            <option backend_node_id="822" value="5">\n                              <text backend_node_id="823">5 guests</text>\n                            </option>\n                            <option backend_node_id="824" value="6">\n                              <text backend_node_id="825">6 guests</text>\n                            </option>\n                            <option backend_node_id="826" value="7">\n                              <text backend_node_id="827">7 guests</text>\n                            </option>\n                            <option backend_node_id="828" value="8">\n                              <text backend_node_id="829">8 guests</text>\n                            </option>\n                            <option backend_node_id="830" value="9">\n                              <text backend_node_id="831">9 guests</text>\n                            </option>\n                            <option backend_node_id="832" value="10">\n                              <text backend_node_id="833">10 guests</text>\n                            </option>\n                          </select>\n                        </div>\n                      </div>\n                      <a backend_node_id="836" aria_label="Search"/>\n                      <span backend_node_id="847">\n                              <text backend_node_id="848">Search</text>\n                            </span>\n                          </div>\n                  </div>\n                </div>\n                <div backend_node_id="853">\n                          <h2 backend_node_id="855">\n                              <text backend_node_id="856">Explore all that Tock has to offer</text>\n                            </h2>\n                          <ul backend_node_id="857">\n                            <li backend_node_id="99">\n                              <div backend_node_id="859">\n                                <img backend_node_id="103" alt="Multiple dishes served with wine"/>\n                                  <p backend_node_id="866">\n                                      <text backend_node_id="867">Dine in</text>\n                                    </p>\n                                  </div>\n                            </li>\n                            <li backend_node_id="100">\n                              <div backend_node_id="869">\n                                <img backend_node_id="104" alt="Several bagged orders of food to-go"/>\n                                  <p backend_node_id="876">\n                                      <text backend_node_id="877">Pickup</text>\n                                    </p>\n                                  </div>\n                            </li>\n                            <li backend_node_id="96">\n                              <div backend_node_id="879">\n                                <img backend_node_id="105" alt="Opened container revealing a hamburger"/>\n                                  <p backend_node_id="885">\n                                      <text backend_node_id="886">Delivery</text>\n                                    </p>\n                                  </div>\n                            </li>\n                            <li backend_node_id="97">\n                              <div backend_node_id="888">\n                                <img backend_node_id="106" alt="Chef in a kitchen"/>\n                                  <p backend_node_id="895">\n                                      <text backend_node_id="896">Events</text>\n                                    </p>\n                                  </div>\n                            </li>\n                            <li backend_node_id="98">\n                              <div backend_node_id="898">\n                                <img backend_node_id="107" alt="People strolling a vineyard sipping wine"/>\n                                  <p backend_node_id="905">\n                                      <text backend_node_id="906">Wineries</text>\n                                    </p>\n                                  </div>\n                            </li>\n                          </ul>\n                        </div>\n                      <div backend_node_id="908">\n                  <div backend_node_id="909">\n                    <div backend_node_id="910">\n                      <section backend_node_id="911">\n                        <h2 backend_node_id="912">\n                          <text backend_node_id="913">New &amp; Notable</text>\n                        </h2>\n                        <div backend_node_id="914">\n                          <p backend_node_id="915">\n                            <text backend_node_id="916">The latest &amp; greatest on Tock</text>\n                          </p>\n                          <a backend_node_id="917" aria_label="Explore all New &amp; Notable">\n                            <span backend_node_id="918">\n                              <text backend_node_id="919">Explore all</text>\n                            </span>\n                          </a>\n                        </div>\n                      </section>\n                      <a backend_node_id="925" aria_label="Explore all New &amp; Notable">\n                        <span backend_node_id="926">\n                          <text backend_node_id="927">Explore all</text>\n                        </span>\n                      </a>\n                    </div>\n                    <div backend_node_id="933">\n                      <div backend_node_id="934">\n                        <ul backend_node_id="935">\n                          <li backend_node_id="936">\n                            <a backend_node_id="937">\n                              <div backend_node_id="938" aria_label="Image of Agni showing font, astronomical object, science, midnight, rectangle, and electric blue" role="img"/>\n                              <section backend_node_id="940">\n                                  <h3 backend_node_id="942">\n                                      <text backend_node_id="943">Agni</text>\n                                    </h3>\n                                  <p backend_node_id="944">\n                                    <text backend_node_id="945">Columbus, OH - Brewery District</text>\n                                    <text backend_node_id="947">Grill</text>\n                                  </p>\n                                </section>\n                              </a>\n                          </li>\n                          <li backend_node_id="949">\n                            <a backend_node_id="950">\n                              <div backend_node_id="951" aria_label="Image of Streetside 62 Bistro showing rectangle, font, signage, circle, brand, and sign" role="img"/>\n                              <section backend_node_id="953">\n                                  <h3 backend_node_id="955">\n                                      <text backend_node_id="956">Streetside 62 Bistro</text>\n                                    </h3>\n                                  <p backend_node_id="957">\n                                    <text backend_node_id="958">Washington Court House, OH</text>\n                                    <text backend_node_id="960">Restaurant</text>\n                                  </p>\n                                </section>\n                              </a>\n                          </li>\n                          <li backend_node_id="962">\n                            <a backend_node_id="963">\n                              <div backend_node_id="964" aria_label="Image of Hell\'s Backbone Grill &amp; Farm showing plant, sky, cloud, building, tree, and window" role="img"/>\n                              <section backend_node_id="966">\n                                  <h3 backend_node_id="968">\n                                      <text backend_node_id="969">Hell\'s Backbone Grill &amp; Farm</text>\n                                    </h3>\n                                  <p backend_node_id="970">\n                                    <text backend_node_id="971">Boulder, UT</text>\n                                    <text backend_node_id="973">Four Corners Farm To Table</text>\n                                  </p>\n                                </section>\n                              </a>\n                          </li>\n                          <li backend_node_id="975">\n                            <a backend_node_id="976">\n                              <div backend_node_id="977" aria_label="Image of Symposium showing plant, building, flowerpot, window, facade, and real estate" role="img"/>\n                              <section backend_node_id="979">\n                                  <h3 backend_node_id="981">\n                                      <text backend_node_id="982">Symposium</text>\n                                    </h3>\n                                  <p backend_node_id="983">\n                                    <text backend_node_id="984">Cincinnati, OH - East Walnut HIlls</text>\n                                    <text backend_node_id="986">Wine Shop</text>\n                                  </p>\n                                </section>\n                              </a>\n                          </li>\n                          <li backend_node_id="988">\n                            <a backend_node_id="989">\n                              <div backend_node_id="990" aria_label="Image of The Merchant Tavern showing font, rectangle, logo, graphics, brand, and grass" role="img"/>\n                              <section backend_node_id="992">\n                                  <h3 backend_node_id="994">\n                                      <text backend_node_id="995">The Merchant Tavern</text>\n                                    </h3>\n                                  <p backend_node_id="996">\n                                    <text backend_node_id="997">Akron, OH - Merriman Valley</text>\n                                    <text backend_node_id="999">American</text>\n                                  </p>\n                                </section>\n                              </a>\n                          </li>\n                          <li backend_node_id="1001">\n                            <a backend_node_id="1002">\n                              <div backend_node_id="1003" aria_label="Image of Luigi\'s Ristorante Italiano showing food, tableware, recipe, condiment, ingredient, and plate" role="img"/>\n                              <section backend_node_id="1005">\n                                  <h3 backend_node_id="1007">\n                                      <text backend_node_id="1008">Luigi\'s Ristorante Italiano</text>\n                                    </h3>\n                                  <p backend_node_id="1009">\n                                    <text backend_node_id="1010">Mason, OH</text>\n                                    <text backend_node_id="1012">Italian</text>\n                                  </p>\n                                </section>\n                              </a>\n                          </li>\n                          <li backend_node_id="1014">\n                            <a backend_node_id="1015">\n                              <div backend_node_id="1016" aria_label="Image of Urban Grill on Main showing plant, cloud, window, sky, building, and tree" role="img"/>\n                              <section backend_node_id="1018">\n                                  <h3 backend_node_id="1020">\n                                      <text backend_node_id="1021">Urban Grill on Main</text>\n                                    </h3>\n                                  <p backend_node_id="1022">\n                                    <text backend_node_id="1023">Cincinnati, OH - Village of Newtown</text>\n                                    <text backend_node_id="1025">American</text>\n                                  </p>\n                                </section>\n                              </a>\n                          </li>\n                          <li backend_node_id="1027">\n                            <a backend_node_id="1028">\n                              <div backend_node_id="1029" aria_label="Image of The Pickle and Pig showing drinking establishment, drinkware, bottle, barware, drink, and alcoholic beverage" role="img"/>\n                              <section backend_node_id="1031">\n                                  <h3 backend_node_id="1033">\n                                      <text backend_node_id="1034">The Pickle and Pig</text>\n                                    </h3>\n                                  <p backend_node_id="1035">\n                                    <text backend_node_id="1036">Oxford, OH - Mile Square</text>\n                                    <text backend_node_id="1038">American</text>\n                                  </p>\n                                </section>\n                              </a>\n                          </li>\n                          <li backend_node_id="1040">\n                            <h2 backend_node_id="1042">\n                                <text backend_node_id="1043">View All</text>\n                              </h2>\n                            </li>\n                        </ul>\n                      </div>\n                    </div>\n                  </div>\n                </div>\n                <div backend_node_id="1051">\n                      <div backend_node_id="1052">\n                        <div backend_node_id="1053">\n                          <div backend_node_id="1054">\n                            <div backend_node_id="1057">\n                              <div backend_node_id="1058">\n                                <h3 backend_node_id="1059">\n                                  <text backend_node_id="1060">Women\'s</text>\n                                </h3>\n                                <h3 backend_node_id="1061">\n                                  <text backend_node_id="1062">History Month</text>\n                                </h3>\n                              </div>\n                              <div backend_node_id="1063">\n                                <p backend_node_id="1064">\n                                  <text backend_node_id="1065">Celebrating and supporting leading women</text>\n                                  <text backend_node_id="1067">shaking up the industry.</text>\n                                </p>\n                                <a backend_node_id="1068">\n                                  <span backend_node_id="1070">\n                                    <text backend_node_id="1071">Explore now</text>\n                                  </span>\n                                </a>\n                              </div>\n                            </div>\n                          </div>\n                        </div>\n                      </div>\n                    </div>\n                  <div backend_node_id="1072">\n                  <div backend_node_id="1073">\n                    <div backend_node_id="1074">\n                      <section backend_node_id="1075">\n                        <h2 backend_node_id="1076">\n                          <text backend_node_id="1077">Tock To Go</text>\n                        </h2>\n                        <div backend_node_id="1078">\n                          <p backend_node_id="1079">\n                            <text backend_node_id="1080">Pickup and delivery meals</text>\n                          </p>\n                          <a backend_node_id="1081" aria_label="Explore all Tock To Go">\n                            <span backend_node_id="1082">\n                              <text backend_node_id="1083">Explore all</text>\n                            </span>\n                          </a>\n                        </div>\n                      </section>\n                      <a backend_node_id="1089" aria_label="Explore all Tock To Go">\n                        <span backend_node_id="1090">\n                          <text backend_node_id="1091">Explore all</text>\n                        </span>\n                      </a>\n                    </div>\n                    <li backend_node_id="1116">\n                            <h2 backend_node_id="1118">\n                                <text backend_node_id="1119">View All</text>\n                              </h2>\n                            </li>\n                        </div>\n                </div>\n                <div backend_node_id="1125">\n                  <div backend_node_id="1126">\n                    <div backend_node_id="1127">\n                      <div backend_node_id="1128">\n                        <div backend_node_id="1129">\n                          <div backend_node_id="1130">\n                            <h3 backend_node_id="1131">\n                              <text backend_node_id="1132">Tock Gift Cards</text>\n                            </h3>\n                            <h4 backend_node_id="1133">\n                              <text backend_node_id="1134">Give the delicious</text>\n                              <text backend_node_id="1136">gift of Tock.</text>\n                            </h4>\n                            <p backend_node_id="1137">\n                              <text backend_node_id="1138">Share your love of great food and wine. For every occasion. Any amount. Never expires. Sure to delight.</text>\n                            </p>\n                            <a backend_node_id="1139">\n                              <span backend_node_id="1141">\n                                <text backend_node_id="1142">Send a gift card</text>\n                              </span>\n                            </a>\n                          </div>\n                        </div>\n                      </div>\n                      <div backend_node_id="1145">\n                        <div backend_node_id="1146">\n                          <div backend_node_id="1147">\n                            <h3 backend_node_id="1148">\n                              <text backend_node_id="1149">The Tock Blog</text>\n                            </h3>\n                            <h4 backend_node_id="1150">\n                              <text backend_node_id="1151">Chef Interviews, Stories, &amp; Curated City Guides.</text>\n                            </h4>\n                            <a backend_node_id="1152">\n                              <span backend_node_id="1154">\n                                <text backend_node_id="1155">Read the latest</text>\n                              </span>\n                            </a>\n                          </div>\n                        </div>\n                      </div>\n                    </div>\n                  </div>\n                </div>\n                <div backend_node_id="1158">\n                  <div backend_node_id="1159">\n                    <div backend_node_id="1160">\n                      <section backend_node_id="1161">\n                        <h2 backend_node_id="1162">\n                          <text backend_node_id="1163">Wineries &amp; Tasting Rooms</text>\n                        </h2>\n                        <div backend_node_id="1164">\n                          <p backend_node_id="1165">\n                            <text backend_node_id="1166">Taste your way through varietals &amp; vintages</text>\n                          </p>\n                          <a backend_node_id="1167" aria_label="Explore all Wineries &amp; Tasting Rooms">\n                            <span backend_node_id="1168">\n                              <text backend_node_id="1169">Explore all</text>\n                            </span>\n                          </a>\n                        </div>\n                      </section>\n                      <a backend_node_id="1175" aria_label="Explore all Wineries &amp; Tasting Rooms">\n                        <span backend_node_id="1176">\n                          <text backend_node_id="1177">Explore all</text>\n                        </span>\n                      </a>\n                    </div>\n                    <li backend_node_id="1202">\n                            <h2 backend_node_id="1204">\n                                <text backend_node_id="1205">View All</text>\n                              </h2>\n                            </li>\n                        </div>\n                </div>\n                <div backend_node_id="1211">\n                  <div backend_node_id="1212">\n                    <div backend_node_id="1213">\n                      <div backend_node_id="1214">\n                        <div backend_node_id="1215">\n                          <div backend_node_id="1216">\n                            <h3 backend_node_id="1217">\n                              <text backend_node_id="1218">Reservations. Events. To-Go.</text>\n                              <text backend_node_id="1220">The</text>\n                              <span backend_node_id="1221">\n                                <text backend_node_id="1223">Only</text>\n                              </span>\n                              <text backend_node_id="1224">All In One Solution.</text>\n                            </h3>\n                          </div>\n                          <div backend_node_id="1225">\n                            <p backend_node_id="1226">\n                              <text backend_node_id="1227">Tock is here to meet the ever-changing needs of hospitality.</text>\n                            </p>\n                            <a backend_node_id="1228">\n                              <span backend_node_id="1230">\n                                <text backend_node_id="1231">Learn more</text>\n                              </span>\n                            </a>\n                          </div>\n                        </div>\n                      </div>\n                    </div>\n                  </div>\n                </div>\n                <div backend_node_id="1232">\n                  <div backend_node_id="1233">\n                    <div backend_node_id="1234">\n                      <section backend_node_id="1235">\n                        <h2 backend_node_id="1236">\n                          <text backend_node_id="1237">Chase Cardmember Tables</text>\n                        </h2>\n                        <div backend_node_id="1238">\n                          <p backend_node_id="1239">\n                            <text backend_node_id="1240">Primetime reservations at restaurants across the country</text>\n                          </p>\n                          <a backend_node_id="1241" aria_label="Explore all Chase Cardmember Tables">\n                            <span backend_node_id="1242">\n                              <text backend_node_id="1243">Explore all</text>\n                            </span>\n                          </a>\n                        </div>\n                      </section>\n                      <a backend_node_id="1249" aria_label="Explore all Chase Cardmember Tables">\n                        <span backend_node_id="1250">\n                          <text backend_node_id="1251">Explore all</text>\n                        </span>\n                      </a>\n                    </div>\n                    <li backend_node_id="1276">\n                            <h2 backend_node_id="1278">\n                                <text backend_node_id="1279">View All</text>\n                              </h2>\n                            </li>\n                        </div>\n                </div>\n                <div backend_node_id="1285">\n                  <div backend_node_id="1286">\n                    <h2 backend_node_id="1289">\n                          <text backend_node_id="1290">Browse all of Tock</text>\n                        </h2>\n                      <button backend_node_id="1344" type="button">\n                          <span backend_node_id="1345">\n                            <text backend_node_id="1346">Load more</text>\n                          </span>\n                        </button>\n                      </div>\n                </div>\n              </div>\n            </main>\n            <footer backend_node_id="1348">\n              <div backend_node_id="1349">\n                <div backend_node_id="1350">\n                  <div backend_node_id="1351">\n                    <div backend_node_id="1352">\n                      <h3 backend_node_id="1353">\n                        <text backend_node_id="1354">Buy a Tock gift card</text>\n                      </h3>\n                      <p backend_node_id="1355">\n                        <text backend_node_id="1356">Give a whole world of experiences, events, and wineries.</text>\n                        <a backend_node_id="1358" aria_label="Learn more about Tock gift cards">\n                          <text backend_node_id="1359">Learn more</text>\n                        </a>\n                      </p>\n                    </div>\n                    <div backend_node_id="1360">\n                      <h3 backend_node_id="1361">\n                        <text backend_node_id="1362">Download the Tock app</text>\n                      </h3>\n                      <p backend_node_id="1363">\n                        <text backend_node_id="1364">Everything you love about Tock is now just a tap away.</text>\n                        <a backend_node_id="1366" aria_label="Download the Tock app">\n                          <text backend_node_id="1367">Download it now</text>\n                        </a>\n                      </p>\n                    </div>\n                    <div backend_node_id="1368">\n                      <h3 backend_node_id="1369">\n                        <text backend_node_id="1370">Use Tock at your business</text>\n                      </h3>\n                      <p backend_node_id="1371">\n                        <text backend_node_id="1372">Curious about our reservation and table management system?</text>\n                        <a backend_node_id="1374" aria_label="Learn more about Tock reservation and table management system">\n                          <text backend_node_id="1375">Learn more</text>\n                        </a>\n                      </p>\n                    </div>\n                  </div>\n                </div>\n              </div>\n              <div backend_node_id="1376">\n                <div backend_node_id="1377">\n                  <a backend_node_id="1378">\n                    <span backend_node_id="1379">\n                      <text backend_node_id="1380">Tock home page</text>\n                    </span>\n                  </a>\n                  <ul backend_node_id="1388">\n                    <li backend_node_id="1389">\n                      <a backend_node_id="1390">\n                        <span backend_node_id="1391">\n                          <text backend_node_id="1392">Support</text>\n                        </span>\n                      </a>\n                    </li>\n                    <li backend_node_id="1393">\n                      <a backend_node_id="1394">\n                        <span backend_node_id="1395">\n                          <text backend_node_id="1396">Careers</text>\n                        </span>\n                      </a>\n                    </li>\n                    <li backend_node_id="1397">\n                      <a backend_node_id="1398">\n                        <span backend_node_id="1399">\n                          <text backend_node_id="1400">Terms</text>\n                        </span>\n                      </a>\n                    </li>\n                    <li backend_node_id="1401">\n                      <a backend_node_id="1402">\n                        <span backend_node_id="1403">\n                          <text backend_node_id="1404">Privacy</text>\n                        </span>\n                      </a>\n                    </li>\n                  </ul>\n                  <ul backend_node_id="1406">\n                    <li backend_node_id="1407">\n                      <a backend_node_id="1408">\n                        <span backend_node_id="1409">\n                          <text backend_node_id="1410">Instagram</text>\n                        </span>\n                      </a>\n                    </li>\n                    <li backend_node_id="1411">\n                      <a backend_node_id="1412">\n                        <span backend_node_id="1413">\n                          <text backend_node_id="1414">Twitter</text>\n                        </span>\n                      </a>\n                    </li>\n                    <li backend_node_id="1415">\n                      <a backend_node_id="1416">\n                        <span backend_node_id="1417">\n                          <text backend_node_id="1418">LinkedIn</text>\n                        </span>\n                      </a>\n                    </li>\n                    <li backend_node_id="1419">\n                      <a backend_node_id="1420">\n                        <span backend_node_id="1421">\n                          <text backend_node_id="1422">Facebook</text>\n                        </span>\n                      </a>\n                    </li>\n                  </ul>\n                  <span backend_node_id="1424">\n                    <text backend_node_id="1425">Explore Tock 2023</text>\n                  </span>\n                </div>\n              </div>\n            </footer>\n          </div>\n        </div>\n      </div>\n    </div>\n    <a backend_node_id="1511">\n                        <text backend_node_id="1512">null</text>\n                      </a>\n                    </body>\n</html>"""
# Simulate a simplified observation (HTML converted to Markdown, typically compressed)
# For the first step, the "Action History" is empty.
CURRENT_CONTEXT = f"""
Current Task: {TASK_INSTRUCTION}
Webpage Snapshot (Simplified HTML/Markdown): {PAGE_HTML}

Action History: (None, this is the first step)
"""

FULL_PROMPT = AGENT_SYSTEM_PROMPT + "\n" + CURRENT_CONTEXT



/home/awais_goated/miniconda3/envs/insta/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

  2025-10-04T11:32:34.016267Z  WARN  Reqwest(reqwest::Error { kind: Request, url: "https://transfer.xethub.hf.co/xorbs/default/a9708fd957d29073eefaf7ab4faf02ae3ac46b7e195223a7976a7ec25afaf607?X-Xet-Signed-Range=bytes%3D0-29907820&X-Xet-Session-Id=01K6QDCG24MWD0VXWJ1Q36NRQH&Expires=1759579921&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly90cmFuc2Zlci54ZXRodWIuaGYuY28veG9yYnMvZGVmYXVsdC9hOTcwOGZkOTU3ZDI5MDczZWVmYWY3YWI0ZmFmMDJhZTNhYzQ2YjdlMTk1MjIzYTc5NzZhN2VjMjVhZmFmNjA3P1gtWGV0LVNpZ25lZC1SYW5nZT1ieXRlcyUzRDAtMjk5MDc4MjAmWC1YZXQtU2Vzc2lvbi1JZD0wMUs2UURDRzI0TVdEMFZYV0oxUTM2TlJRSCIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc1OTU3OTkyMX19fV19&Signature=m2ASPfEaYqdhVy7nVsRzU6EmfEauu4xP2rIGiGwELtuiyjCL1KluetS6g-TrlyhOW5FloedTGq~Lvd~HK3~SwJRM3x-Qfmg9UNiMDH1lBca4doFZc8hGfHDSCCSeS40PRkVt6Ph-ul2BzuirSYjmmRWStBQxeUjMEfo7qUimUoBqTuyd9UoOqvso1iyFRKd4hTjHeYUCMNqSk32w9~N2AXEf4NXIS-MVd398f4OcBAKZEj95TWrzOaga7BeBSqzu8Jai9p-txB-cC0HLQfW7LtnADaO1CEtGH4odgSzSiGKxKZqYgRTJJx2W30ZP2

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  4.76it/s]


In [2]:
# 4. Generate the Agent's Response
# We limit generation length to ensure it completes the reasoning trace and the JSON action [3].
input_ids = tokenizer(FULL_PROMPT, return_tensors="pt").input_ids.to(model.device)
# Max new tokens is set to 1024 tokens in the InSTA pipeline [19]
output = model.generate(
    input_ids,
    max_new_tokens=1024,
    do_sample=False,  # Use deterministic sampling for action generation
    pad_token_id=tokenizer.eos_token_id
)

response_text = tokenizer.decode(output[0], skip_special_tokens=True)

# 5. Display Result
print("--- FULL AGENT PROMPT ---")
print(FULL_PROMPT)
print("\n" + "="*50)
print("--- MODEL RESPONSE (Expected Format) ---")

# The response will contain the reasoning and the JSON action [2]
# Example expected output based on source [2, 20]:
print(f"""
#### I’m viewing the Adobe fonts homepage, and I will type into the search box to browse fonts suitable for a children’s book.
#### ```json {{     “action_key”: “fill”,     “action_kwargs”: {{         “text”: “children’s book”     }},     “target_element_id”: 13 }} ```
""")
print("\n" + "="*50)
print("--- ACTUAL DECODED RESPONSE (If run live) ---")
# In a real environment, you would parse this decoded response
print(response_text)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--- FULL AGENT PROMPT ---

You are helping me complete tasks by operating a web browser. I will share the current task, and a sequence of webpages and actions from previous steps.

## Action Instructions
Based on the information we discovered so far, and the progress we made in previous steps, you are helping me determine the next action.
You will provide an action as JSON in a fenced code block:
```json {
"action_key": str, "action_kwargs": dict, "target_element_id": int | null
} ```
... [Action Definitions Omitted for Brevity] ...
## Formatting Your Response
Write a 200 word revised plan based on new information we discovered, and progress we made in previous steps. After your response, provide the next action as JSON in a fenced code block.


Current Task: Book a winery tour in Napa Valley in a winery which serves Mediterranean cuisine with wine testing for 4 guests on April 15, 10 am in a outdoor setup
Webpage Snapshot (Simplified HTML/Markdown): <html backend_node_id="117">
  <bod

In [8]:
print(type(response_text))

<class 'str'>


In [9]:
# Example: Convert raw HTML to markdown using MarkdownProcessor
# ...save as run_markdown_processor_example.py...

from insta.observation_processors.markdown_processor import MarkdownProcessor
from insta.configs.browser_config import BrowserObservation

# Example raw HTML and dummy metadata
raw_html = """
<html>
  <body>
    <h1>Welcome to Tock</h1>
    <p id= "p1">Book a winery tour in Napa Valley.</p>
    <div id='test'>Outter div
        <div>Inner div</div>
    </div>
    <button id="book">Book Now</button>
  </body>
</html>
"""

# Minimal metadata (empty dict for demo; real usage should provide DOM node metadata)
metadata = {}

# Create a BrowserObservation object
observation = BrowserObservation(
    raw_html=raw_html,
    metadata=metadata,
    screenshot=None,
    current_url="https://www.example.com",
    processed_text=None
)

# Instantiate the MarkdownProcessor
processor = MarkdownProcessor()

# Process the observation to get markdown
markdown_obs = processor.process(observation, remove_pii=False)

print("--- Markdown Output ---")
print(markdown_obs.processed_text)

--- Markdown Output ---

# Welcome to Tock
 Book a winery tour in Napa Valley. Outter div Inner div Book Now


In [6]:
markdown_obs

BrowserObservation(processed_text='\n# Welcome to {{ORGANIZATION}} a winery tour in Napa Valley. Book Now', raw_html='\n<html>\n  <body>\n    <h1>Welcome to Tock</h1>\n    <p id= "p1">Book a winery tour in Napa Valley.</p>\n    <button id="book">Book Now</button>\n  </body>\n</html>\n', processed_image=None, screenshot=None, metadata={}, current_url='https://www.example.com')